In [2]:
import os
from dotenv import load_dotenv

#load_dotenv(dotenv_path='.env')
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError(".env 파일에 GROQ_API_KEY 가 없습니다.")
print("GROQ_API_KEY 설정 확인 완료", GROQ_API_KEY[:2])


GROQ_API_KEY 설정 확인 완료 gs


#### Groq API로 호출 가능한 모델 목록 확인

- Groq는 OpenAI 호환 API이므로 `openai` SDK에 `base_url`만 Groq 주소로 지정하면 `client.models.list()`로 모델 목록을 조회할 수 있습니다.
- 목록 조회는 토큰을 사용하지 않습니다.
- 아래 `ChatOpenAI`에서 사용할 모델명이 목록에 있는지 확인하는 용도로도 사용할 수 있습니다.

In [3]:
# openai SDK 불러오기 (Groq도 OpenAI 호환 API라서 같은 SDK 사용)
from openai import OpenAI

GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트

# Groq 키와 엔드포인트로 클라이언트 생성 (GROQ_API_KEY는 load_dotenv()로 .env에서 읽어 둔 값)
client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL)

# 호출 가능한 모델 목록 조회 (id 기준 오름차순 정렬)
models = sorted(client.models.list().data, key=lambda m: m.id)
print(f"조회된 모델 수: {len(models)}\n")
for m in models:
    print(f"{m.id:<45} {m.owned_by}")

조회된 모델 수: 11

allam-2-7b                                    SDAIA
canopylabs/orpheus-arabic-saudi               Canopy Labs
canopylabs/orpheus-v1-english                 Canopy Labs
meta-llama/llama-prompt-guard-2-22m           Meta
meta-llama/llama-prompt-guard-2-86m           Meta
openai/gpt-oss-120b                           OpenAI
openai/gpt-oss-20b                            OpenAI
openai/gpt-oss-safeguard-20b                  OpenAI
qwen/qwen3.8-27b                              Alibaba Cloud
whisper-large-v3                              OpenAI
whisper-large-v3-turbo                        OpenAI


In [4]:
from langchain_core.prompts import ChatPromptTemplate

# prompt
prompt = ChatPromptTemplate.from_messages(
    [ ("system", "당신은 개발자입니다.") , 
     ("user", "{input}") ]
)
print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]


In [5]:
prompt_text = prompt.format(input="파이썬은 무엇인가요? 자세하게 설명해주세요")
print(type(prompt_text))
print(prompt_text)

<class 'str'>
System: 당신은 개발자입니다.
Human: 파이썬은 무엇인가요? 자세하게 설명해주세요


In [6]:
from langchain_openai import ChatOpenAI 

#llm = ChatOpenAI(api_key=OPENAI_API_KEY, model_name="gpt-3.5-turbo-0125")

# Groq API를 사용하는 ChatOpenAI 인스턴스 생성
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",
    # model="qwen/qwen3.8-27b",
    temperature=0.7
)
print(type(llm))
print(llm.model_name)

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
openai/gpt-oss-120b


In [7]:
try:
    response = llm.invoke(prompt_text)
    print(type(response))
    print("응답:", response.content)
except Exception as e:
    print(f"오류 발생: {e}")

<class 'langchain_core.messages.ai.AIMessage'>
응답: ## 파이썬(Python)이란?

**파이썬**은 1991년에 귀도 반 로썸(Guido van Rossum)이 발표한 고수준(high‑level) 프로그래밍 언어입니다. 현재는 전 세계 개발자 커뮤니티가 활발히 유지·보수하고 있으며, **오픈 소스**로 누구나 자유롭게 사용하고 수정할 수 있습니다. 파이썬은 **읽기 쉬운 문법**, **풍부한 표준 라이브러리**, **다양한 분야에서의 활용성** 덕분에 초·중·고등학생부터 연구원, 기업 개발자까지 널리 쓰이고 있습니다.

---

## 1. 파이썬의 주요 특징

| 특징 | 설명 |
|------|------|
| **인터프리터 언어** | 코드를 컴파일하지 않고 바로 실행한다. (REPL, Jupyter Notebook 등) |
| **동적 타이핑** | 변수에 타입을 명시하지 않아도 런타임에 자동으로 결정한다. |
| **읽기 쉬운 문법** | 들여쓰기로 블록을 구분하고, 불필요한 기호(세미콜론, 중괄호 등)를 최소화한다. |
| **풍부한 표준 라이브러리** | `os`, `datetime`, `json`, `re`, `urllib` 등 수백 개의 모듈이 기본 제공된다. |
| **확장성** | C/C++ 로 작성된 확장 모듈을 쉽게 연결할 수 있다 (예: NumPy, SciPy). |
| **멀티패러다임** | 절차적, 객체지향, 함수형 프로그래밍을 모두 지원한다. |
| **플랫폼 독립성** | Windows, macOS, Linux, 심지어 임베디드 시스템까지 대부분의 OS에서 동일하게 동작한다. |
| **대규모 커뮤니티** | PyPI(Python Package Index)에는 300,000개가 넘는 서드파티 패키지가 존재한다. |
| **버전** | 현재 주류는 Python 3.x (특히 3.10, 3.11, 3.12 등). Python 2는 2020년 공식 지원 종료. |

---

## 2. 파이썬의 

## LCEL(LangChain Expression Language)
### Prompt + LLM 모델 + OutputParser

In [8]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [9]:
chain = prompt | llm | output_parser
print(type(chain))
print(chain)

<class 'langchain_core.runnables.base.RunnableSequence'>
first=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]) middle=[ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001F801DD46B0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001F8020221B0>, root_client=<openai.OpenAI object at 0x000001F87FEF4FE0>, root_async_client=<openai.AsyncOpenAI object at 0x000001F802021A60>, model_name='openai/gpt-oss-120b', temperature=0.7, model_kwargs={}, ope

In [10]:
response = chain.invoke({"input":"LangChain은 무엇인가요?"})

In [11]:
print(type(response))
print(response)

<class 'langchain_core.messages.base.TextAccessor'>
**LangChain(랑체인)**은 **LLM(대규모 언어 모델)**을 활용한 **애플리케이션을 보다 쉽게 설계·구현·운영**할 수 있도록 도와주는 **오픈소스 프레임워크**입니다.  
주요 목표는 “**LLM을 단독으로 쓰는 것이 아니라, 외부 도구·데이터·워크플로와 결합**”해서 실제 비즈니스 로직이나 복잡한 작업을 수행하는 애플리케이션을 빠르게 만들게 하는 것”입니다.

아래에서는 LangChain의 핵심 개념, 주요 구성 요소, 사용 흐름, 그리고 실제 활용 사례를 단계별로 정리해 보겠습니다.

---

## 1️⃣ LangChain의 핵심 아이디어

| 요소 | 설명 |
|------|------|
| **LLM** | GPT‑4, Claude, LLaMA 등 텍스트 생성·이해를 담당하는 모델. |
| **프롬프트 체이닝** | 여러 프롬프트를 순차·조건부로 연결해 복잡한 작업을 단계별로 수행. |
| **외부 도구 연동** | 검색 엔진, 데이터베이스, API, 파일 시스템 등과 연결해 LLM이 “생각”만 하는 것이 아니라 실제 행동을 수행하게 함. |
| **메모리** | 이전 대화·작업 결과를 저장·활용해 컨텍스트를 유지하고 연속적인 상호작용을 가능하게 함. |
| **체인(Chain)** | 프롬프트·도구·메모리 등을 조합한 **작업 흐름**. 하나의 체인은 여러 “스텝”(Component)으로 구성됩니다. |
| **에이전트(Agent)** | LLM이 “어떤 도구를 언제 사용할지”를 스스로 판단하도록 만든 **동적 체인**. “LLM이 계획을 세우고, 도구를 호출하고, 결과를 종합”하는 루프를 구현. |

> **핵심 포인트**: LangChain은 “LLM + 도구 + 흐름(Chain)”이라는 삼각형 구조를 제공해, LLM을 **‘스마트한 엔진’**이 아니라 **‘스마트한 시스템’**으로 변환합니다.

---

## 2️⃣ 주요 구성 요소

| 구